# Fine Tuning with Optuna Framework

In [1]:
%load_ext autoreload
%autoreload 2

In [16]:
#!pip freeze > requirements.txt

In [2]:
!pip install optuna --quiet
!pip install optuna-integration[sklearn]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 kB 2.4 MB/s eta 0:00:00


In [3]:
# Data Science Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Train model
# from ml_pipeline import *

# Utils
import warnings
import os
import json
import joblib


pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=FutureWarning)

In [4]:
# Configurar conexion con google drive y con carpeta de google colab
from google.colab import drive
drive.mount('/content/drive')
workin_dir = '/content/drive/MyDrive/Colab Notebooks/Prediccion de Adiciones Proyectos de Infraestructura'
os.chdir(workin_dir)
import sys
# Add the directory containing the .py file to the Python path
sys.path.append(workin_dir) # Replace with the actual path if different

# Now you can import functions from your .py file
from ml_pipeline import *

Mounted at /content/drive


In [5]:
# Load list of columns
with open('Data/projectsColumns.txt', 'r') as f:
    projectsColumns = f.read().splitlines()
with open('Data/ownerColumns.txt', 'r') as f:
    ownerColumns = f.read().splitlines()
with open('Data/contractorColumns.txt', 'r') as f:
    contractorColumns = f.read().splitlines()
with open('Data/terridataColumns.txt', 'r') as f:
    terridataColumns = f.read().splitlines()
with open('Data/outcomeColumns.txt', 'r') as f:
    outcomeColumns = f.read().splitlines()

In [6]:
df = pd.read_csv('Data/trainData.csv')   # Load the data
df = df[df['contractValueMw(G1)'] > 1e3].copy().reset_index()  # Filter out rows with contractValueMw(G1) <= 1000
# convert contractMonthSigned(G2) and contractMonthStart(G2) to object
#df['contractMonthSigned(G2)'] = df['contractMonthSigned(G2)'].astype(str)
#df['contractMonthStart(G2)'] = df['contractMonthStart(G2)'].astype(str)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6994 entries, 0 to 6993
Data columns (total 46 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   index                                  6994 non-null   int64  
 1   contractValueMw(G1)                    6994 non-null   float64
 2   contractDurationDays(G1)               6994 non-null   float64
 3   projectIntensityMw(G1)                 6994 non-null   float64
 4   workCategory(G1)                       6994 non-null   object 
 5   prebidValueMw(G1)                      6994 non-null   float64
 6   bidContractRatio(G1)                   6994 non-null   float64
 7   contractMethodType(G1)                 6994 non-null   object 
 8   contractMonthSigned(G1)                6994 non-null   int64  
 9   contractMonthStart(G1)                 6994 non-null   int64  
 10  electoralYear(G1)                      6994 non-null   int64  
 11  pree

In [7]:
vars_s1 = projectsColumns
vars_s2 = vars_s1 + ownerColumns
vars_s3 = vars_s2 + contractorColumns
vars_s4 = vars_s3 + terridataColumns
set_vars = [vars_s1, vars_s2, vars_s3, vars_s4]

## Cost Deviation

### Logistic Regression

In [8]:
param_suggest_dict_lr = {
    'penalty': lambda trial: trial.suggest_categorical('penalty', ['l1', 'l2']),
    'C': lambda trial: trial.suggest_loguniform('C', 1e-4, 1e4),
}
lr_model = LogisticRegression(max_iter=10000, solver='liblinear', random_state=42)

In [24]:
lr_results = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveCostDeviation(G5)']

  # Split data into training and testing sets
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

  lr_train_tuning_evaluate = train_tuning_evaluate_model(
    X_train=X_train,
    y_train=y_train,
    model=lr_model,
    model_name='Logistic Regression',
    stage=stage,
    param_suggest_dict=param_suggest_dict_lr,
    X_test=X_test,
    y_test=y_test,
    n_trials=50,  # Reducido para el ejemplo
    cv=5,          # Reducido para el ejemplo
    scoring='roc_auc',
    direction='maximize',
    timeout=None,
    resampling_strategy='smoteenn',  # Ejemplo con SMOTE
    random_state=42,
    plot_tuning_history=False)

  lr_results[stage] = lr_train_tuning_evaluate

[I 2025-05-28 03:55:44,547] A new study created in memory with name: no-name-f7d789a7-b9ee-4279-be11-4a63279369d4


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 03:55:50,359] Trial 0 finished with value: 0.571248955756365 and parameters: {'penalty': 'l2', 'C': 71.77141927992021}. Best is trial 0 with value: 0.571248955756365.
[I 2025-05-28 03:55:51,513] Trial 1 finished with value: 0.5 and parameters: {'penalty': 'l1', 'C': 0.0017699302940633358}. Best is trial 0 with value: 0.571248955756365.
[I 2025-05-28 03:55:52,563] Trial 2 finished with value: 0.571501590148232 and parameters: {'penalty': 'l2', 'C': 6.4405075539937195}. Best is trial 2 with value: 0.571501590148232.
[I 2025-05-28 03:55:56,044] Trial 3 finished with value: 0.5709925220069104 and parameters: {'penalty': 'l1', 'C': 5744.851636320435}. Best is trial 2 with value: 0.571501590148232.
[I 2025-05-28 03:55:57,076] Trial 4 finished with value: 0.5 and parameters: {'penalty': 'l1', 'C': 0.002848391870910803}. Best is trial 2 with value: 0.571501590148232.
[I 2025-05-28 03:55:58,197] Trial 5 finished with value: 0.5718395586425551 and parameters: {'penalty': 'l2', 'C':

[I 2025-05-28 03:57:19,316] A new study created in memory with name: no-name-746e4608-c5d6-4436-93ab-55bb1782c021



===== Logistic Regression Tuning Report in S1 =====
Best roc_auc: 0.5721

Best parameters:
- penalty: l2
- C: 0.5385959148305941

Parameter importance:
- C: 0.8643
- penalty: 0.1357

===== Logistic Regression Performance Report =====

Test Set Metrics:
- accuracy: 0.4340
- precision: 0.3534
- recall: 0.7833
- f1: 0.4870
- roc_auc: 0.5670


...

[I 2025-05-28 03:59:43,526] Trial 0 finished with value: 0.6961499623248094 and parameters: {'penalty': 'l2', 'C': 71.77141927992021}. Best is trial 0 with value: 0.6961499623248094.
[I 2025-05-28 04:00:05,873] Trial 1 finished with value: 0.5213591424127971 and parameters: {'penalty': 'l1', 'C': 0.0017699302940633358}. Best is trial 0 with value: 0.6961499623248094.
[I 2025-05-28 04:00:26,879] Trial 2 finished with value: 0.6990559930609791 and parameters: {'penalty': 'l2', 'C': 6.4405075539937195}. Best is trial 2 with value: 0.6990559930609791.
[I 2025-05-28 04:00:52,572] Trial 3 finished with value: 0.6956503304022184 and parameters: {'penalty': 'l1', 'C': 5744.851636320435}. Best is trial 2 with value: 0.6990559930609791.
[I 2025-05-28 04:01:14,717] Trial 4 finished with value: 0.6567368799314575 and parameters: {'penalty': 'l1', 'C': 0.002848391870910803}. Best is trial 2 with value: 0.6990559930609791.
[I 2025-05-28 04:01:34,443] Trial 5 finished with value: 0.7017946878245578 a

[I 2025-05-28 04:14:02,392] A new study created in memory with name: no-name-440748fe-b834-4672-884f-bd5ea52747a2



===== Logistic Regression Tuning Report in S3 =====
Best roc_auc: 0.7058

Best parameters:
- penalty: l2
- C: 0.08528933855762823

Parameter importance:
- C: 0.8552
- penalty: 0.1448

===== Logistic Regression Performance Report =====

Test Set Metrics:
- accuracy: 0.5755
- precision: 0.4351
- recall: 0.7958
- f1: 0.5626
- roc_auc: 0.6885


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 04:14:05,492] Trial 0 finished with value: 0.7015575177059459 and parameters: {'penalty': 'l2', 'C': 71.77141927992021}. Best is trial 0 with value: 0.7015575177059459.
[I 2025-05-28 04:14:09,116] Trial 1 finished with value: 0.5213591424127971 and parameters: {'penalty': 'l1', 'C': 0.0017699302940633358}. Best is trial 0 with value: 0.7015575177059459.
[I 2025-05-28 04:14:11,505] Trial 2 finished with value: 0.7048160661824231 and parameters: {'penalty': 'l2', 'C': 6.4405075539937195}. Best is trial 2 with value: 0.7048160661824231.
[I 2025-05-28 04:14:18,506] Trial 3 finished with value: 0.700418156289253 and parameters: {'penalty': 'l1', 'C': 5744.851636320435}. Best is trial 2 with value: 0.7048160661824231.
[I 2025-05-28 04:14:21,731] Trial 4 finished with value: 0.6675082040076369 and parameters: {'penalty': 'l1', 'C': 0.002848391870910803}. Best is trial 2 with value: 0.7048160661824231.
[I 2025-05-28 04:14:25,021] Trial 5 finished with value: 0.7074922348834635 an

In [25]:
# Guardar los resultados
joblib.dump(lr_results, 'Models/cost_deviation_lr_results.joblib')
print("Diccionario complejo guardado exitosamente")

Diccionario complejo guardado exitosamente


In [26]:
# Cargar el diccionario
lr_results = joblib.load('Models/cost_deviation_lr_results.joblib')
# print("Diccionario complejo cargado exitosamente")

### Random Forest

In [10]:
param_suggest_dict_rf = {
    # Profundidad máxima: ampliando el rango para explorar más posibilidades
    'max_depth': lambda trial: trial.suggest_int('max_depth', 5, 50),

    # Número de estimadores: mantener el rango original pero con exploración continua
    'n_estimators': lambda trial: trial.suggest_int('n_estimators', 50, 500),

    # Samples para dividir: explorar valores con distribución logarítmica
    'min_samples_split': lambda trial: trial.suggest_int('min_samples_split', 2, 20),

    # Samples en hojas: explorar valores con distribución logarítmica
    'min_samples_leaf': lambda trial: trial.suggest_int('min_samples_leaf', 1, 10),

    # Características a considerar: explorar valores continuos y categóricos
    'max_features': lambda trial: trial.suggest_categorical('max_features', ['sqrt', 'log2'])
                    if trial.suggest_categorical('max_features_type', ['auto', 'float']) == 'auto'
                    else trial.suggest_float('max_features_float', 0.1, 0.9),

    # Parámetros adicionales que podrían mejorar el rendimiento
    'bootstrap': lambda trial: trial.suggest_categorical('bootstrap', [True, False]),
}

In [ ]:
rf_results = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveCostDeviation(G5)']

  # Split data into training and testing sets
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  rf_model = RandomForestClassifier(random_state=42)
  # Usar la función para optimizar el modelo
  rf_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=rf_model,
      model_name='Random Forest',
      stage=stage,
      param_suggest_dict=param_suggest_dict_rf,
      X_test=X_test,
      y_test=y_test,
      n_trials=50,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False
  )
  rf_results[stage] = rf_train_tuning_evaluate

In [28]:
# Guardar los resultados del random forest
joblib.dump(rf_results, 'Models/cost_deviation_rf_results.joblib')
print("Diccionario complejo guardado exitosamente")

Diccionario complejo guardado exitosamente


In [29]:
rf_results = joblib.load('Models/cost_deviation_rf_results.joblib')

### Extreme Gradient Boosting

In [11]:
# Definir el diccionario de sugerencias de parámetros para Optuna
param_suggest_dict_xgb = {
    'max_depth': lambda trial: trial.suggest_int('max_depth', 3, 10),
    'learning_rate': lambda trial: trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
    'n_estimators': lambda trial: trial.suggest_int('n_estimators', 50, 300),
    'min_child_weight': lambda trial: trial.suggest_int('min_child_weight', 1, 10),
    'subsample': lambda trial: trial.suggest_float('subsample', 0.5, 1.0),
    'colsample_bytree': lambda trial: trial.suggest_float('colsample_bytree', 0.5, 1.0),
    'gamma': lambda trial: trial.suggest_float('gamma', 0, 5)
}

In [30]:
# Usar la función para optimizar el modelo
xgb_results = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveCostDeviation(G5)']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  # Definir el modelo base
  xgb_model = XGBClassifier(random_state=42)
  xgb_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=xgb_model,
      model_name='XGBoost',
      stage=stage,
      param_suggest_dict=param_suggest_dict_xgb,
      X_test=X_test,
      y_test=y_test,
      n_trials=50,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False
      )
  xgb_results[stage] = xgb_train_tuning_evaluate

[I 2025-05-28 05:31:52,241] A new study created in memory with name: no-name-0dc6df19-8ebc-4f85-a976-840f9ff20fb5


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 05:31:54,148] Trial 0 finished with value: 0.5744413253900892 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'n_estimators': 233, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'gamma': 0.2904180608409973}. Best is trial 0 with value: 0.5744413253900892.
[I 2025-05-28 05:31:57,154] Trial 1 finished with value: 0.5880048270091445 and parameters: {'max_depth': 9, 'learning_rate': 0.07725378389307355, 'n_estimators': 227, 'min_child_weight': 1, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'gamma': 1.0616955533913808}. Best is trial 1 with value: 0.5880048270091445.
[I 2025-05-28 05:31:59,470] Trial 2 finished with value: 0.5885237946498105 and parameters: {'max_depth': 4, 'learning_rate': 0.018659959624904916, 'n_estimators': 126, 'min_child_weight': 6, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021, 'gamma': 3.0592644736118975}. Best is trial 2 with va

[I 2025-05-28 05:33:40,654] A new study created in memory with name: no-name-3b323783-b5f9-479f-88c2-3e13ad4438b5



===== XGBoost Tuning Report in S1 =====
Best roc_auc: 0.5981

Best parameters:
- max_depth: 5
- learning_rate: 0.01243057262076087
- n_estimators: 275
- min_child_weight: 1
- subsample: 0.5126642976394511
- colsample_bytree: 0.5909721083751994
- gamma: 0.2663271650204529

Parameter importance:
- learning_rate: 0.6209
- min_child_weight: 0.1067
- n_estimators: 0.0837
- gamma: 0.0759
- subsample: 0.0635
- colsample_bytree: 0.0371
- max_depth: 0.0121

===== XGBoost Performance Report =====

Test Set Metrics:
- accuracy: 0.5250
- precision: 0.3864
- recall: 0.6542
- f1: 0.4858
- roc_auc: 0.5982


...

[I 2025-05-28 05:38:39,283] Trial 0 finished with value: 0.7111708727884778 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'n_estimators': 233, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'gamma': 0.2904180608409973}. Best is trial 0 with value: 0.7111708727884778.
[I 2025-05-28 05:38:59,831] Trial 1 finished with value: 0.7135192786850596 and parameters: {'max_depth': 9, 'learning_rate': 0.07725378389307355, 'n_estimators': 227, 'min_child_weight': 1, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'gamma': 1.0616955533913808}. Best is trial 1 with value: 0.7135192786850596.
[I 2025-05-28 05:39:16,016] Trial 2 finished with value: 0.7056972498792898 and parameters: {'max_depth': 4, 'learning_rate': 0.018659959624904916, 'n_estimators': 126, 'min_child_weight': 6, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021, 'gamma': 3.0592644736118975}. Best is trial 1 with va

[I 2025-05-28 05:53:01,726] A new study created in memory with name: no-name-0bd197ea-2eae-4a3b-921b-3848ae39b275



===== XGBoost Tuning Report in S3 =====
Best roc_auc: 0.7195

Best parameters:
- max_depth: 7
- learning_rate: 0.07087191450507684
- n_estimators: 226
- min_child_weight: 6
- subsample: 0.6024614133397443
- colsample_bytree: 0.6679850819417708
- gamma: 1.296931177353891

Parameter importance:
- learning_rate: 0.5367
- n_estimators: 0.2349
- gamma: 0.1168
- colsample_bytree: 0.0420
- max_depth: 0.0373
- subsample: 0.0193
- min_child_weight: 0.0130

===== XGBoost Performance Report =====

Test Set Metrics:
- accuracy: 0.6246
- precision: 0.4702
- recall: 0.7444
- f1: 0.5763
- roc_auc: 0.7118


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 05:53:05,831] Trial 0 finished with value: 0.7175051631879531 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'n_estimators': 233, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'gamma': 0.2904180608409973}. Best is trial 0 with value: 0.7175051631879531.
[I 2025-05-28 05:53:14,495] Trial 1 finished with value: 0.7201784257522892 and parameters: {'max_depth': 9, 'learning_rate': 0.07725378389307355, 'n_estimators': 227, 'min_child_weight': 1, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'gamma': 1.0616955533913808}. Best is trial 1 with value: 0.7201784257522892.
[I 2025-05-28 05:53:18,217] Trial 2 finished with value: 0.7162139986749406 and parameters: {'max_depth': 4, 'learning_rate': 0.018659959624904916, 'n_estimators': 126, 'min_child_weight': 6, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021, 'gamma': 3.0592644736118975}. Best is trial 1 with va

In [31]:
# Guardar los resultados del extreme gradient boosting
joblib.dump(xgb_results, 'Models/cost_deviation_xgb_results.joblib')
print("Diccionario complejo guardado exitosamente")

Diccionario complejo guardado exitosamente


In [32]:
xgb_results = joblib.load('Models/cost_deviation_xgb_results.joblib')

### K Nearest Neighbors

In [12]:
param_suggest_dict_knn = {
    # Número de vecinos: explorar un rango más amplio y continuo
    'n_neighbors': lambda trial: trial.suggest_int('n_neighbors', 3, 40),

    # Función de pesos
    'weights': lambda trial: trial.suggest_categorical('weights', ['uniform', 'distance']),

    # Métrica de distancia con manejo condicional para el parámetro p
    'metric': lambda trial: trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),

    # Parámetro p solo relevante cuando metric='minkowski'
    #'p': lambda trial: trial.suggest_float('p', 1.1, 1.9) if trial.params.get('metric') == 'minkowski' else 2,

    # # Algoritmo para calcular vecinos
    # 'algorithm': lambda trial: trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute'])
    #                 if trial.params.get('metric') in ['euclidean', 'manhattan'] else 'auto',

    # # Tamaño de hoja - importante para ball_tree y kd_tree
    # 'leaf_size': lambda trial: trial.suggest_int('leaf_size', 10, 100)
    #              if trial.params.get('algorithm') in ['ball_tree', 'kd_tree'] else 30
}

In [ ]:
knn_results = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveCostDeviation(G5)']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  knn_model = KNeighborsClassifier()

  # Usar la función para optimizar el modelo
  knn_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=knn_model,
      model_name='KNN',
      stage=stage,
      param_suggest_dict=param_suggest_dict_knn,
      X_test=X_test,
      y_test=y_test,
      n_trials=50,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False)
  knn_results[stage] = knn_train_tuning_evaluate

[I 2025-05-28 05:58:08,815] A new study created in memory with name: no-name-ab7e9372-b3e4-4f3b-bbd0-0d93357aa6ae


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 05:58:09,921] Trial 0 finished with value: 0.5363819320704707 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.5363819320704707.
[I 2025-05-28 05:58:11,056] Trial 1 finished with value: 0.5495690881753246 and parameters: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 1 with value: 0.5495690881753246.
[I 2025-05-28 05:58:12,182] Trial 2 finished with value: 0.5420620545639797 and parameters: {'n_neighbors': 34, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 1 with value: 0.5495690881753246.
[I 2025-05-28 05:58:13,713] Trial 3 finished with value: 0.5496158082110529 and parameters: {'n_neighbors': 19, 'weights': 'distance', 'metric': 'cosine'}. Best is trial 3 with value: 0.5496158082110529.
[I 2025-05-28 05:58:15,676] Trial 4 finished with value: 0.5466979692858409 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 3 with value:

[I 2025-05-28 05:59:22,210] A new study created in memory with name: no-name-cb4831e8-33ea-4db8-9cf7-fd13312ce8da



===== KNN Tuning Report in S1 =====
Best roc_auc: 0.5626

Best parameters:
- n_neighbors: 40
- weights: distance
- metric: manhattan

Parameter importance:
- metric: 0.4162
- n_neighbors: 0.3189
- weights: 0.2649

===== KNN Performance Report =====

Test Set Metrics:
- accuracy: 0.4888
- precision: 0.3726
- recall: 0.7167
- f1: 0.4903
- roc_auc: 0.5686


...

[I 2025-05-28 05:59:24,038] Trial 0 finished with value: 0.6687126274556335 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.6687126274556335.
[I 2025-05-28 05:59:25,913] Trial 1 finished with value: 0.6644193130730726 and parameters: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 0 with value: 0.6687126274556335.
[I 2025-05-28 05:59:27,823] Trial 2 finished with value: 0.6783432077823646 and parameters: {'n_neighbors': 34, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 2 with value: 0.6783432077823646.
[I 2025-05-28 05:59:30,363] Trial 3 finished with value: 0.6860388578176899 and parameters: {'n_neighbors': 19, 'weights': 'distance', 'metric': 'cosine'}. Best is trial 3 with value: 0.6860388578176899.
[I 2025-05-28 05:59:34,257] Trial 4 finished with value: 0.6975813602937662 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 4 with value:

[I 2025-05-28 06:01:41,259] A new study created in memory with name: no-name-a2f03d3f-8a2e-4e26-ab20-8eac2a1695e6



===== KNN Tuning Report in S2 =====
Best roc_auc: 0.7048

Best parameters:
- n_neighbors: 20
- weights: distance
- metric: manhattan

Parameter importance:
- metric: 0.8858
- n_neighbors: 0.0609
- weights: 0.0533

===== KNN Performance Report =====

Test Set Metrics:
- accuracy: 0.5788
- precision: 0.4405
- recall: 0.8431
- f1: 0.5786
- roc_auc: 0.7110


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 06:01:57,416] Trial 0 finished with value: 0.6764720620764304 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.6764720620764304.
[I 2025-05-28 06:02:13,468] Trial 1 finished with value: 0.6664662744928987 and parameters: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 0 with value: 0.6764720620764304.
[I 2025-05-28 06:02:29,578] Trial 2 finished with value: 0.6852692186604242 and parameters: {'n_neighbors': 34, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 2 with value: 0.6852692186604242.
[I 2025-05-28 06:02:45,687] Trial 3 finished with value: 0.689277836773168 and parameters: {'n_neighbors': 19, 'weights': 'distance', 'metric': 'cosine'}. Best is trial 3 with value: 0.689277836773168.
[I 2025-05-28 06:03:03,936] Trial 4 finished with value: 0.6949802422712514 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 4 with value: 0

In [ ]:
# Guardar los resultados del KNN
joblib.dump(knn_results, 'Models/cost_deviation_knn_results.joblib')
print("Diccionario complejo guardado exitosamente")

In [ ]:
knn_results = joblib.load('Models/cost_deviation_knn_results.joblib')

### Cost Deviation: results

In [ ]:
lr_metrics = {}
for i in np.arange(1, len(set_vars)+1):
  lr_metrics[str(i)] = lr_results[f'S{i}']['metrics']
lr_metrics = pd.DataFrame(lr_metrics).T

rf_metrics = {}
for i in np.arange(1, len(set_vars)+1):
  rf_metrics[str(i)] = rf_results[f'S{i}']['metrics']
rf_metrics = pd.DataFrame(rf_metrics).T

xgb_metrics = {}
for i in np.arange(1, len(set_vars)+1):
  xgb_metrics[str(i)] = xgb_results[f'S{i}']['metrics']
xgb_metrics = pd.DataFrame(xgb_metrics).T

knn_metrics = {}
for i in np.arange(1, len(set_vars)+1):
  knn_metrics[str(i)] = knn_results[f'S{i}']['metrics']
knn_metrics = pd.DataFrame(knn_metrics).T

metrics_cost_deviation = pd.concat([lr_metrics, rf_metrics, xgb_metrics, knn_metrics], axis=0)
#metrics_cost_deviation.to_excel('Data/metrics_cost_deviation.xlsx', index=False)
metrics_cost_deviation

In [ ]:
sns.lineplot(data=metrics_cost_deviation, x='stage', y='roc_auc', hue='model')
plt.title('Metric: ROC AUC')
plt.xlabel('Settings')
plt.ylabel('ROC AUC')
plt.legend(title='Model', loc='lower right')
plt.show()

In [ ]:
sns.lineplot(data=metrics_cost_deviation, x='stage', y='f1', hue='model')
plt.title('Metric: F1')
plt.xlabel('Settings')
plt.ylabel('F1')
plt.legend(title='Model', loc='lower right')
plt.show()

In [ ]:
sns.lineplot(data=metrics_cost_deviation, x='stage', y='precision', hue='model')
plt.title('Metric: Precision')
plt.xlabel('Settings')
plt.ylabel('Precision')
plt.legend(title='Model', loc='lower right')
plt.show()

In [ ]:
sns.lineplot(data=metrics_cost_deviation, x='stage', y='recall', hue='model')
plt.title('Metric: Recall')
plt.xlabel('Settings')
plt.ylabel('Recall')
plt.legend(title='Model', loc='lower right')
plt.show()

In [ ]:
sns.lineplot(data=metrics_cost_deviation, x='stage', y='accuracy', hue='model')
plt.title('Metric: Accuracy')
plt.xlabel('Settings')
plt.ylabel('Accuracy')
plt.legend(title='Model', loc='lower right')

In [ ]:
metrics_cost_deviation.set_index(['model', 'stage']).to_latex(
    index=True,
    float_format=lambda x: "{:.2f}\%".format(x * 100)
)

In [ ]:
# metrics_cost_deviation_tidy = metrics_cost_deviation.melt(id_vars=['model', 'stage'], var_name='metric', value_name='value')
# metrics_cost_deviation_tidy

In [ ]:
# in two columns and two rows plot de the metric for each model, stage and metric
# fig, ax = plt.subplots(3, 2, figsize=(8,8))
# for i, metric in enumerate(metrics_cost_deviation_tidy['metric'].unique()):
#  sns.lineplot(data = metrics_cost_deviation_tidy.query(f'metric == "{metric}"'), x='stage', y='value',
#               hue='model', ax=ax[i//2, i%2], markers=True)
#  name_metric = metric.title() if metric != 'roc_auc' else 'ROC AUC'
#  ax[i//2, i%2].set_title(f'Metric: {name_metric}')
#  ax[i//2, i%2].set_xlabel(None)
#  ax[i//2, i%2].set_ylabel('Value')
#  # legend out of plot
#  ax[i//2, i%2].legend(loc='lower right', fontsize='xx-small')
#  # Set grid only in yaxis
#  ax[i//2, i%2].grid(axis='x')
# plt.tight_layout()
# plt.savefig('Data/all_results_cost_deviation.png')
# plt.show()

In [ ]:
0.48*2

## Time Deviation

### Logistic Regression

In [ ]:
param_suggest_dict_lr = {
    'penalty': lambda trial: trial.suggest_categorical('penalty', ['l1', 'l2']),
    'C': lambda trial: trial.suggest_loguniform('C', 1e-4, 1e4),
}
lr_model = LogisticRegression(max_iter=10000, solver='liblinear', random_state=42)

lr_results_time_deviation = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveTimeDeviation(G5)']

  # Split data into training and testing sets
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

  lr_train_tuning_evaluate = train_tuning_evaluate_model(
    X_train=X_train,
    y_train=y_train,
    model=lr_model,
    model_name='Logistic Regression',
    stage=stage,
    param_suggest_dict=param_suggest_dict_lr,
    X_test=X_test,
    y_test=y_test,
    n_trials=50,  # Reducido para el ejemplo
    cv=5,          # Reducido para el ejemplo
    scoring='roc_auc',
    direction='maximize',
    timeout=None,
    resampling_strategy='smoteenn',  # Ejemplo con SMOTE
    random_state=42,
    plot_tuning_history=False)

  lr_results_time_deviation[stage] = lr_train_tuning_evaluate

In [ ]:
# Guardar los resultados
joblib.dump(lr_results_time_deviation, 'Models/time_deviation_lr_results.joblib')
print("Diccionario complejo guardado exitosamente")

In [ ]:
lr_results_time_deviation = joblib.load('Models/time_deviation_lr_results.joblib')

In [ ]:
# Cargar el diccionario
# loaded_dict = joblib.load('Models/cost_deviation_lr_results.joblib')
# print("Diccionario complejo cargado exitosamente")

### Random Forest

In [13]:
param_suggest_dict_rf = {
    # Profundidad máxima: ampliando el rango para explorar más posibilidades
    'max_depth': lambda trial: trial.suggest_int('max_depth', 5, 50),

    # Número de estimadores: mantener el rango original pero con exploración continua
    'n_estimators': lambda trial: trial.suggest_int('n_estimators', 50, 500),

    # Samples para dividir: explorar valores con distribución logarítmica
    'min_samples_split': lambda trial: trial.suggest_int('min_samples_split', 2, 20),

    # Samples en hojas: explorar valores con distribución logarítmica
    'min_samples_leaf': lambda trial: trial.suggest_int('min_samples_leaf', 1, 10),

    # Características a considerar: explorar valores continuos y categóricos
    'max_features': lambda trial: trial.suggest_categorical('max_features', ['sqrt', 'log2'])
                    if trial.suggest_categorical('max_features_type', ['auto', 'float']) == 'auto'
                    else trial.suggest_float('max_features_float', 0.1, 0.9),

    # Parámetros adicionales que podrían mejorar el rendimiento
    'bootstrap': lambda trial: trial.suggest_categorical('bootstrap', [True, False]),
}



rf_results_time_deviation = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveTimeDeviation(G5)']

  # Split data into training and testing sets
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  rf_model = RandomForestClassifier(random_state=42)
  # Usar la función para optimizar el modelo
  rf_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=rf_model,
      model_name='Random Forest',
      stage=stage,
      param_suggest_dict=param_suggest_dict_rf,
      X_test=X_test,
      y_test=y_test,
      n_trials=50,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False
  )
  rf_results_time_deviation[stage] = rf_train_tuning_evaluate

[I 2025-05-28 11:24:01,013] A new study created in memory with name: no-name-67e381e6-528f-449b-bed5-f21efaa3671b


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 11:24:10,960] Trial 0 finished with value: 0.6229535044634258 and parameters: {'max_depth': 22, 'n_estimators': 478, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features_type': 'auto', 'max_features': 'log2', 'bootstrap': False}. Best is trial 0 with value: 0.6229535044634258.
[I 2025-05-28 11:24:19,245] Trial 1 finished with value: 0.6311613402824235 and parameters: {'max_depth': 5, 'n_estimators': 487, 'min_samples_split': 17, 'min_samples_leaf': 3, 'max_features_type': 'float', 'max_features_float': 0.3433937943676302, 'bootstrap': True}. Best is trial 1 with value: 0.6311613402824235.
[I 2025-05-28 11:24:28,952] Trial 2 finished with value: 0.6177459506542691 and parameters: {'max_depth': 18, 'n_estimators': 325, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features_type': 'float', 'max_features_float': 0.728140769114411, 'bootstrap': False}. Best is trial 1 with value: 0.6311613402824235.
[I 2025-05-28 11:24:32,979] Trial 3 finished with value: 0.6235

[I 2025-05-28 11:29:28,808] A new study created in memory with name: no-name-8f77357d-5fe7-4fd9-b514-bfa5ee29beab



===== Random Forest Tuning Report in S1 =====
Best roc_auc: 0.6312

Best parameters:
- max_depth: 5
- n_estimators: 487
- min_samples_split: 17
- min_samples_leaf: 3
- max_features_type: float
- max_features_float: 0.3433937943676302
- bootstrap: True

Parameter importance:
- bootstrap: 0.8145
- max_features_type: 0.0862
- max_depth: 0.0362
- n_estimators: 0.0269
- min_samples_split: 0.0190
- min_samples_leaf: 0.0171

===== Random Forest Performance Report =====

Test Set Metrics:
- accuracy: 0.5436
- precision: 0.4962
- recall: 0.9101
- f1: 0.6423
- roc_auc: 0.6292


...

[I 2025-05-28 11:39:15,708] Trial 0 finished with value: 0.7659334560577741 and parameters: {'max_depth': 22, 'n_estimators': 478, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features_type': 'auto', 'max_features': 'log2', 'bootstrap': False}. Best is trial 0 with value: 0.7659334560577741.
[I 2025-05-28 11:39:44,435] Trial 1 finished with value: 0.753519405314534 and parameters: {'max_depth': 5, 'n_estimators': 487, 'min_samples_split': 17, 'min_samples_leaf': 3, 'max_features_type': 'float', 'max_features_float': 0.3433937943676302, 'bootstrap': True}. Best is trial 0 with value: 0.7659334560577741.
[I 2025-05-28 11:40:47,640] Trial 2 finished with value: 0.7564346599290837 and parameters: {'max_depth': 18, 'n_estimators': 325, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features_type': 'float', 'max_features_float': 0.728140769114411, 'bootstrap': False}. Best is trial 0 with value: 0.7659334560577741.
[I 2025-05-28 11:41:09,635] Trial 3 finished with value: 0.75728

[I 2025-05-28 12:01:10,297] A new study created in memory with name: no-name-492865b1-9491-4b56-8c8c-1b7f7f64827d



===== Random Forest Tuning Report in S3 =====
Best roc_auc: 0.7729

Best parameters:
- max_depth: 20
- n_estimators: 485
- min_samples_split: 7
- min_samples_leaf: 1
- max_features_type: auto
- max_features: log2
- bootstrap: False

Parameter importance:
- max_features_type: 0.4896
- max_depth: 0.1877
- min_samples_leaf: 0.1529
- min_samples_split: 0.0843
- n_estimators: 0.0649
- bootstrap: 0.0206

===== Random Forest Performance Report =====

Test Set Metrics:
- accuracy: 0.6970
- precision: 0.6354
- recall: 0.7672
- f1: 0.6951
- roc_auc: 0.7783


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 12:01:20,423] Trial 0 finished with value: 0.7665152716897998 and parameters: {'max_depth': 22, 'n_estimators': 478, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features_type': 'auto', 'max_features': 'log2', 'bootstrap': False}. Best is trial 0 with value: 0.7665152716897998.
[I 2025-05-28 12:01:39,292] Trial 1 finished with value: 0.7565485243633024 and parameters: {'max_depth': 5, 'n_estimators': 487, 'min_samples_split': 17, 'min_samples_leaf': 3, 'max_features_type': 'float', 'max_features_float': 0.3433937943676302, 'bootstrap': True}. Best is trial 0 with value: 0.7665152716897998.
[I 2025-05-28 12:02:31,312] Trial 2 finished with value: 0.7576386104842873 and parameters: {'max_depth': 18, 'n_estimators': 325, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features_type': 'float', 'max_features_float': 0.728140769114411, 'bootstrap': False}. Best is trial 0 with value: 0.7665152716897998.
[I 2025-05-28 12:02:42,960] Trial 3 finished with value: 0.7570

In [14]:
# Guardar los resultados del random forest
joblib.dump(rf_results_time_deviation, 'Models/time_deviation_rf_results.joblib')
print("Diccionario complejo guardado exitosamente")

Diccionario complejo guardado exitosamente


In [15]:
rf_results_time_deviation = joblib.load('Models/time_deviation_rf_results.joblib')

### Extreme Gradient Boosting

In [16]:
# Definir el diccionario de sugerencias de parámetros para Optuna
param_suggest_dict_xgb = {
    'max_depth': lambda trial: trial.suggest_int('max_depth', 3, 10),
    'learning_rate': lambda trial: trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
    'n_estimators': lambda trial: trial.suggest_int('n_estimators', 50, 300),
    'min_child_weight': lambda trial: trial.suggest_int('min_child_weight', 1, 10),
    'subsample': lambda trial: trial.suggest_float('subsample', 0.5, 1.0),
    'colsample_bytree': lambda trial: trial.suggest_float('colsample_bytree', 0.5, 1.0),
    'gamma': lambda trial: trial.suggest_float('gamma', 0, 5)
}

# Usar la función para optimizar el modelo
xgb_results_time_deviation = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveTimeDeviation(G5)']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  # Definir el modelo base
  xgb_model = XGBClassifier(random_state=42)
  xgb_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=xgb_model,
      model_name='XGBoost',
      stage=stage,
      param_suggest_dict=param_suggest_dict_xgb,
      X_test=X_test,
      y_test=y_test,
      n_trials=50,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False
      )
  xgb_results_time_deviation[stage] = xgb_train_tuning_evaluate

[I 2025-05-28 12:10:21,682] A new study created in memory with name: no-name-b8e823ca-5c61-4fc3-beb6-2d51ebb235c7


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 12:10:24,085] Trial 0 finished with value: 0.6073560950387098 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'n_estimators': 233, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'gamma': 0.2904180608409973}. Best is trial 0 with value: 0.6073560950387098.
[I 2025-05-28 12:10:26,509] Trial 1 finished with value: 0.6234383330584992 and parameters: {'max_depth': 9, 'learning_rate': 0.07725378389307355, 'n_estimators': 227, 'min_child_weight': 1, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'gamma': 1.0616955533913808}. Best is trial 1 with value: 0.6234383330584992.
[I 2025-05-28 12:10:27,900] Trial 2 finished with value: 0.6307868180811502 and parameters: {'max_depth': 4, 'learning_rate': 0.018659959624904916, 'n_estimators': 126, 'min_child_weight': 6, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021, 'gamma': 3.0592644736118975}. Best is trial 2 with va

[I 2025-05-28 12:11:46,860] A new study created in memory with name: no-name-36ec01c7-3272-4bf7-a4cc-14783d694cf7



===== XGBoost Tuning Report in S1 =====
Best roc_auc: 0.6341

Best parameters:
- max_depth: 3
- learning_rate: 0.023518772241124787
- n_estimators: 185
- min_child_weight: 5
- subsample: 0.6366999493298028
- colsample_bytree: 0.7691213881637928
- gamma: 3.922993933260105

Parameter importance:
- gamma: 0.5166
- learning_rate: 0.3048
- subsample: 0.0575
- n_estimators: 0.0380
- max_depth: 0.0365
- min_child_weight: 0.0331
- colsample_bytree: 0.0134

===== XGBoost Performance Report =====

Test Set Metrics:
- accuracy: 0.5560
- precision: 0.5043
- recall: 0.8127
- f1: 0.6224
- roc_auc: 0.6278


...

[I 2025-05-28 12:14:55,900] Trial 0 finished with value: 0.7495524117867574 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'n_estimators': 233, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'gamma': 0.2904180608409973}. Best is trial 0 with value: 0.7495524117867574.
[I 2025-05-28 12:15:09,903] Trial 1 finished with value: 0.7649465304673387 and parameters: {'max_depth': 9, 'learning_rate': 0.07725378389307355, 'n_estimators': 227, 'min_child_weight': 1, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'gamma': 1.0616955533913808}. Best is trial 1 with value: 0.7649465304673387.
[I 2025-05-28 12:15:22,719] Trial 2 finished with value: 0.7504944958617773 and parameters: {'max_depth': 4, 'learning_rate': 0.018659959624904916, 'n_estimators': 126, 'min_child_weight': 6, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021, 'gamma': 3.0592644736118975}. Best is trial 1 with va

[I 2025-05-28 12:26:44,679] A new study created in memory with name: no-name-56b62074-fe4d-46ed-9356-a984dc644281



===== XGBoost Tuning Report in S3 =====
Best roc_auc: 0.7672

Best parameters:
- max_depth: 10
- learning_rate: 0.040359456141514524
- n_estimators: 300
- min_child_weight: 1
- subsample: 0.8699813459549738
- colsample_bytree: 0.9176741985472457
- gamma: 0.6700565316310413

Parameter importance:
- min_child_weight: 0.3053
- max_depth: 0.2359
- gamma: 0.1537
- n_estimators: 0.1460
- subsample: 0.0861
- colsample_bytree: 0.0568
- learning_rate: 0.0161

===== XGBoost Performance Report =====

Test Set Metrics:
- accuracy: 0.6927
- precision: 0.6339
- recall: 0.7513
- f1: 0.6877
- roc_auc: 0.7738


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 12:26:47,629] Trial 0 finished with value: 0.7533255679610626 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'n_estimators': 233, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'gamma': 0.2904180608409973}. Best is trial 0 with value: 0.7533255679610626.
[I 2025-05-28 12:26:51,674] Trial 1 finished with value: 0.7667224700258302 and parameters: {'max_depth': 9, 'learning_rate': 0.07725378389307355, 'n_estimators': 227, 'min_child_weight': 1, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'gamma': 1.0616955533913808}. Best is trial 1 with value: 0.7667224700258302.
[I 2025-05-28 12:26:55,225] Trial 2 finished with value: 0.7552016004615821 and parameters: {'max_depth': 4, 'learning_rate': 0.018659959624904916, 'n_estimators': 126, 'min_child_weight': 6, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021, 'gamma': 3.0592644736118975}. Best is trial 1 with va

In [17]:
# Guardar los resultados del extreme gradient boosting
joblib.dump(xgb_results_time_deviation, 'Models/time_deviation_xgb_results.joblib')
print("Diccionario complejo guardado exitosamente")

Diccionario complejo guardado exitosamente


In [18]:
xgb_results_time_deviation = joblib.load('Models/time_deviation_xgb_results.joblib')

### K Nearest Neighbors

In [19]:
param_suggest_dict_knn = {
    # Número de vecinos: explorar un rango más amplio y continuo
    'n_neighbors': lambda trial: trial.suggest_int('n_neighbors', 3, 40),

    # Función de pesos
    'weights': lambda trial: trial.suggest_categorical('weights', ['uniform', 'distance']),

    # Métrica de distancia con manejo condicional para el parámetro p
    'metric': lambda trial: trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),

    # Parámetro p solo relevante cuando metric='minkowski'
    #'p': lambda trial: trial.suggest_float('p', 1.1, 1.9) if trial.params.get('metric') == 'minkowski' else 2,

    # # Algoritmo para calcular vecinos
    # 'algorithm': lambda trial: trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute'])
    #                 if trial.params.get('metric') in ['euclidean', 'manhattan'] else 'auto',

    # # Tamaño de hoja - importante para ball_tree y kd_tree
    # 'leaf_size': lambda trial: trial.suggest_int('leaf_size', 10, 100)
    #              if trial.params.get('algorithm') in ['ball_tree', 'kd_tree'] else 30
}
knn_results_time_deviation = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveTimeDeviation(G5)']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  knn_model = KNeighborsClassifier()

  # Usar la función para optimizar el modelo
  knn_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=knn_model,
      model_name='KNN',
      stage=stage,
      param_suggest_dict=param_suggest_dict_knn,
      X_test=X_test,
      y_test=y_test,
      n_trials=50,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False)
  knn_results_time_deviation[stage] = knn_train_tuning_evaluate

[I 2025-05-28 12:31:17,265] A new study created in memory with name: no-name-17a7ee47-69bf-42a7-95d2-d2f74f8f91aa


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 12:31:18,179] Trial 0 finished with value: 0.581055866651145 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.581055866651145.
[I 2025-05-28 12:31:19,098] Trial 1 finished with value: 0.5790273302578376 and parameters: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 0 with value: 0.581055866651145.
[I 2025-05-28 12:31:20,032] Trial 2 finished with value: 0.5812973544718101 and parameters: {'n_neighbors': 34, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 2 with value: 0.5812973544718101.
[I 2025-05-28 12:31:20,995] Trial 3 finished with value: 0.5882284710252653 and parameters: {'n_neighbors': 19, 'weights': 'distance', 'metric': 'cosine'}. Best is trial 3 with value: 0.5882284710252653.
[I 2025-05-28 12:31:21,988] Trial 4 finished with value: 0.5967179995289154 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 4 with value: 0.

[I 2025-05-28 12:32:19,582] A new study created in memory with name: no-name-82e3adbb-cfac-4ad5-aaf5-70525edbee1b



===== KNN Tuning Report in S1 =====
Best roc_auc: 0.5986

Best parameters:
- n_neighbors: 39
- weights: distance
- metric: manhattan

Parameter importance:
- metric: 0.9650
- n_neighbors: 0.0185
- weights: 0.0165

===== KNN Performance Report =====

Test Set Metrics:
- accuracy: 0.5450
- precision: 0.4965
- recall: 0.7418
- f1: 0.5948
- roc_auc: 0.5873


...

[I 2025-05-28 12:34:20,568] Trial 0 finished with value: 0.7250003241232179 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.7250003241232179.
[I 2025-05-28 12:34:32,855] Trial 1 finished with value: 0.7088753256994217 and parameters: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 0 with value: 0.7250003241232179.
[I 2025-05-28 12:34:43,778] Trial 2 finished with value: 0.7287491390096035 and parameters: {'n_neighbors': 34, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 2 with value: 0.7287491390096035.
[I 2025-05-28 12:34:56,413] Trial 3 finished with value: 0.7359858356586256 and parameters: {'n_neighbors': 19, 'weights': 'distance', 'metric': 'cosine'}. Best is trial 3 with value: 0.7359858356586256.
[I 2025-05-28 12:35:09,865] Trial 4 finished with value: 0.7430727056970239 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 4 with value:

[I 2025-05-28 12:44:56,095] A new study created in memory with name: no-name-7bacfd16-9030-4018-a343-724e29bc54f6



===== KNN Tuning Report in S3 =====
Best roc_auc: 0.7528

Best parameters:
- n_neighbors: 22
- weights: distance
- metric: manhattan

Parameter importance:
- metric: 0.7771
- n_neighbors: 0.1187
- weights: 0.1042

===== KNN Performance Report =====

Test Set Metrics:
- accuracy: 0.6827
- precision: 0.6247
- recall: 0.7397
- f1: 0.6773
- roc_auc: 0.7563


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-28 12:44:58,032] Trial 0 finished with value: 0.7318214098137334 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.7318214098137334.
[I 2025-05-28 12:44:59,939] Trial 1 finished with value: 0.7196175447044466 and parameters: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 0 with value: 0.7318214098137334.
[I 2025-05-28 12:45:02,767] Trial 2 finished with value: 0.7338573426953914 and parameters: {'n_neighbors': 34, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 2 with value: 0.7338573426953914.
[I 2025-05-28 12:45:05,945] Trial 3 finished with value: 0.7433631052961908 and parameters: {'n_neighbors': 19, 'weights': 'distance', 'metric': 'cosine'}. Best is trial 3 with value: 0.7433631052961908.
[I 2025-05-28 12:45:08,995] Trial 4 finished with value: 0.74625864256012 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 4 with value: 0

In [20]:
# Guardar los resultados del KNN
joblib.dump(knn_results_time_deviation, 'Models/time_deviation_knn_results.joblib')
print("Diccionario complejo guardado exitosamente")

Diccionario complejo guardado exitosamente


In [21]:
knn_results_time_deviation = joblib.load('Models/time_deviation_knn_results.joblib')

### Time Deviation: results

In [23]:
lr_metrics_time_deviation = {}
for i in np.arange(1, len(set_vars)+1):
  lr_metrics_time_deviation[str(i)] = lr_results_time_deviation[f'S{i}']['metrics']
lr_metrics_time_deviation = pd.DataFrame(lr_metrics_time_deviation).T

rf_metrics_time_deviation = {}
for i in np.arange(1, len(set_vars)+1):
  rf_metrics_time_deviation[str(i)] = rf_results_time_deviation[f'S{i}']['metrics']
rf_metrics_time_deviation = pd.DataFrame(rf_metrics_time_deviation).T

xgb_metrics_time_deviation = {}
for i in np.arange(1, len(set_vars)+1):
  xgb_metrics_time_deviation[str(i)] = xgb_results_time_deviation[f'S{i}']['metrics']
xgb_metrics_time_deviation = pd.DataFrame(xgb_metrics_time_deviation).T

knn_metrics_time_deviation = {}
for i in np.arange(1, len(set_vars)+1):
  knn_metrics_time_deviation[str(i)] = knn_results_time_deviation[f'S{i}']['metrics']
knn_metrics_time_deviation = pd.DataFrame(knn_metrics_time_deviation).T

metrics_time_deviation = pd.concat([lr_metrics_time_deviation, rf_metrics_time_deviation, xgb_metrics_time_deviation, knn_metrics_time_deviation], axis=0)
#metrics_time_deviation.to_excel('Data/metrics_time_deviation.xlsx', index=False)
metrics_time_deviation

NameError: name 'lr_results_time_deviation' is not defined

In [ ]:
sns.lineplot(data=metrics_time_deviation, x='stage', y='roc_auc', hue='model')
plt.title('Metric: ROC AUC')
plt.xlabel('Settings')
plt.ylabel('ROC AUC')
plt.legend(title='Model', loc='lower right')
plt.show()

In [ ]:
sns.lineplot(data=metrics_time_deviation, x='stage', y='f1', hue='model')
plt.title('Metric: F1')
plt.xlabel('Settings')
plt.ylabel('F1')
plt.legend(title='Model', loc='lower right')
plt.show()

In [ ]:
sns.lineplot(data=metrics_time_deviation, x='stage', y='precision', hue='model')
plt.title('Metric: Precision')
plt.xlabel('Settings')
plt.ylabel('Precision')
plt.legend(title='Model', loc='lower right')
plt.show()

In [ ]:
sns.lineplot(data=metrics_time_deviation, x='stage', y='recall', hue='model')
plt.title('Metric: Recall')
plt.xlabel('Settings')
plt.ylabel('Recall')
plt.legend(title='Model', loc='upper right')
plt.show()

In [ ]:
sns.lineplot(data=metrics_time_deviation, x='stage', y='accuracy', hue='model')
plt.title('Metric: Accuracy')
plt.xlabel('Settings')
plt.ylabel('Accuracy')
plt.legend(title='Model', loc='lower right')
plt.show()

In [ ]:
metrics_time_deviation.set_index(['model','stage']).to_latex(
    index=True,
    float_format=lambda x: "{:.2f}\%".format(x * 100)
)